# XLS-R Evaluation (300 Utterances)
**Nepali-English Code-Mixed ASR**

This notebook evaluates the fine-tuned Wav2Vec2-XLS-R-300M model on the **300-utterance test set**.

In [ ]:
!pip install -q jiwer


In [ ]:
import os
import torch
import pandas as pd
from transformers import Wav2Vec2Processor, Wav2Vec2ForCTC

class Config:
    model_path = "/kaggle/input/models/aadarsh17elmundo/wav2vec2-xls-r/transformers/default/1/outputs/best_checkpoint"
    csv_path = "/kaggle/input/datasets/panditaadarsh/codeswitchv3/metadata_cycle2.csv"
    audio_dir = "/kaggle/input/datasets/panditaadarsh/nepali-english-codeswitched/kaggle_upload/audios_segment"
    
    test_set_size = 300
    batch_size = 8

config = Config()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")


In [ ]:
import re
import unicodedata
import jiwer
import librosa
from tqdm import tqdm

DEVANAGARI_START = 0x0900
DEVANAGARI_END = 0x097F

# Standard ASR text normalization: Lowercase and strip punctuation
def asr_normalize(text: str) -> str:
    text = unicodedata.normalize('NFC', text)
    text = text.lower()
    # Remove punctuation (keep only alphanumeric and spaces, taking care of Devanagari)
    # \w in python regex matches Unicode word characters (including Devanagari)
    text = re.sub(r'[^\w\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def is_nepali(word: str) -> bool:
    return any(DEVANAGARI_START <= ord(ch) <= DEVANAGARI_END for ch in word)

def is_english(word: str) -> bool:
    # A word is English if it contains at least one Latin letter and no Devanagari characters
    has_latin = any('a' <= ch <= 'z' for ch in word)
    has_devanagari = is_nepali(word)
    return has_latin and not has_devanagari

def evaluate_predictions_corpus_level(references, predictions):
    norm_refs = [asr_normalize(r) for r in references]
    norm_preds = [asr_normalize(p) for p in predictions]
    
    # 1. Overall WER and CER (Micro-Average over entire corpus)
    out_wer = jiwer.process_words(norm_refs, norm_preds)
    overall_wer = out_wer.wer * 100
    
    out_cer = jiwer.process_characters(norm_refs, norm_preds)
    overall_cer = out_cer.cer * 100
    
    # 2. Language-Specific WERs (Micro-Average)
    total_nep_s, total_nep_d, total_nep_i, total_nep_n = 0, 0, 0, 0
    total_en_s,  total_en_d,  total_en_i,  total_en_n = 0, 0, 0, 0
    
    for r, p in zip(norm_refs, norm_preds):
        # Filter sequences for Nepali (Matrix language)
        r_nep = ' '.join(w for w in r.split() if is_nepali(w))
        p_nep = ' '.join(w for w in p.split() if is_nepali(w))
        
        if len(r_nep) == 0 and len(p_nep) > 0:
            total_nep_i += len(p_nep.split())
        elif len(r_nep) > 0:
            out_n = jiwer.process_words(r_nep, p_nep)
            total_nep_s += out_n.substitutions
            total_nep_d += out_n.deletions
            total_nep_i += out_n.insertions
            total_nep_n += len(r_nep.split())
            
        # Filter sequences for English (Embedded language)
        r_en = ' '.join(w for w in r.split() if is_english(w))
        p_en = ' '.join(w for w in p.split() if is_english(w))
        
        if len(r_en) == 0 and len(p_en) > 0:
            total_en_i += len(p_en.split())
        elif len(r_en) > 0:
            out_e = jiwer.process_words(r_en, p_en)
            total_en_s += out_e.substitutions
            total_en_d += out_e.deletions
            total_en_i += out_e.insertions
            total_en_n += len(r_en.split())
            
    nep_wer = ((total_nep_s + total_nep_d + total_nep_i) / total_nep_n) * 100 if total_nep_n > 0 else 0
    cm_wer = ((total_en_s + total_en_d + total_en_i) / total_en_n) * 100 if total_en_n > 0 else 0
    
    return {
        "WER": overall_wer,
        "CER": overall_cer,
        "Nep": nep_wer,
        "CM-WER": cm_wer
    }


In [ ]:
import csv
print('Loading hold-out test set...')
paths, texts = [], []
with open(config.csv_path, 'r', encoding='utf-8') as f:
    reader = csv.reader(f)
    header = next(reader)
    for row in reader:
        if len(row) > 1:
            paths.append(row[0].strip())
            texts.append(','.join(row[1:]).strip())

df = pd.DataFrame({'audio_path': paths, 'text': texts})
df = df[df['text'] != ''].reset_index(drop=True)
df['audio_path'] = df['audio_path'].apply(
    lambda x: x if os.path.isabs(x) else os.path.join(config.audio_dir, os.path.basename(x))
)

mask = df['audio_path'].apply(os.path.exists)
df = df[mask].reset_index(drop=True)

test_df = df.iloc[-config.test_set_size:].reset_index(drop=True)
print(f'Test set ready: {len(test_df)} utterances.')


In [ ]:
def run_inference_ctc(model, processor, df):
    predictions = []
    references = df['text'].tolist()
    audio_paths = df['audio_path'].tolist()
    
    for i in tqdm(range(0, len(audio_paths), config.batch_size), desc='Inference (XLS-R)'):
        batch_paths = audio_paths[i:i + config.batch_size]
        batch_arrays = []
        for path in batch_paths:
            arr, sr = librosa.load(path, sr=16000, mono=True)
            batch_arrays.append(arr)

        inputs = processor(batch_arrays, sampling_rate=16000, return_tensors='pt', padding=True).to(device)
        # NOTE: Removed .half() to avoid layer norm overflow in CTC models

        with torch.no_grad():
            logits = model(**inputs).logits
            predicted_ids = torch.argmax(logits, dim=-1)

        preds = processor.batch_decode(predicted_ids)
        predictions.extend(preds)

    return predictions, references


In [ ]:
print("Loading Fine-Tuned XLS-R Model...")
processor = Wav2Vec2Processor.from_pretrained(config.model_path)
model = Wav2Vec2ForCTC.from_pretrained(config.model_path).to(device)
model.eval()

xls_r_preds, refs = run_inference_ctc(model, processor, test_df)
xls_r_metrics = evaluate_predictions_corpus_level(refs, xls_r_preds)


In [ ]:
import json
import base64
import zlib

final_results = {
    "XLS_R_300M_CS": xls_r_metrics,
}

print("\n================ SUMMARY OF ALL METRICS ================")
for model_name, metrics in final_results.items():
    print(f"{model_name}:")
    print(f"  WER:    {metrics['WER']:.2f}%")
    print(f"  CER:    {metrics['CER']:.2f}%")
    print(f"  Nep:    {metrics['Nep']:.2f}%")
    print(f"  CM-WER: {metrics['CM-WER']:.2f}%")
    print()

# Create zipped string
json_str = json.dumps(final_results)
compressed = zlib.compress(json_str.encode("utf-8"))
b64_str = base64.b64encode(compressed).decode("utf-8")

print("\n--- COPY THIS ENTIRE STRING AND PASTE IT TO THE ASSISTANT ---")
print(f"ZIPPED_RESULTS_BEGIN:{b64_str}:ZIPPED_RESULTS_END")
